In [385]:
import sys
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')

from lambda_cox import LambdaSA
from utils import concordance_index, unroll, pad_to, get_targets_and_masks
import yaml
import jax
import jax.numpy as jnp
import haiku as hk
import numpy as np
import matplotlib.pyplot as plt

In [384]:
from importlib import reload
import utils
reload(utils)

<module 'utils' from '/Users/mariana/Documents/projects/Huawei/survan/utils.py'>

In [2]:
from functools import partial

In [20]:
config_path = '../config.yaml'
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

In [21]:
agent = LambdaSA(config, 42)

In [22]:
xs = agent.data['seqs']
ts = agent.data['ts']
cs = agent.data['cs']

n, H, _ = xs.shape
# xs = jnp.concatenate((xs, jnp.tile(jnp.eye(H), (n, 1, 1))), axis=-1)
# agent.integrated_brier_score(xs[:,0], ts, cs)

In [23]:
xs.shape

(467, 5, 5)

In [24]:
agent.lambda_ = 0.0

In [26]:
# ys, ws = agent._targets(3, xs, ts, cs)

In [ ]:
ys[(ts == 2 & cs)]

In [27]:
#Construct target

base_tgt = agent.forward(agent.state.params, xs)
# base_tgt = base_tgt.squeeze(1)
base_tgt = jax.nn.sigmoid(base_tgt)

In [28]:
base_tgt.shape

(467, 5, 5)

In [29]:
n, T, _ = base_tgt.shape

In [30]:
#Move things one step
b_tgt = base_tgt[0]
h_tgt = jnp.hstack((jnp.zeros((T, 1)), base_tgt[0][:, 1:]))

In [31]:
h_tgt

Array([[0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5]], dtype=float32)

In [32]:
b_tgt

Array([[0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5]], dtype=float32)

In [33]:
h = jnp.zeros_like(h_tgt)
h.shape

(5, 5)

In [34]:
h = h.at[T-1, 1:].set(b_tgt[T-1, 1:])

In [35]:
h

Array([[0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. ],
       [0. , 0.5, 0.5, 0.5, 0.5]], dtype=float32)

In [36]:
# Loop version:

lambda_ = 1.0
h = jnp.zeros_like(h_tgt)
h = h.at[T-1, 1:].set(b_tgt[T-1, 1:])

In [46]:
h_tgt

Array([[0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5],
       [0. , 0.5, 0.5, 0.5, 0.5]], dtype=float32)

In [38]:
for i in reversed(range(T-1)):
    tmp = lambda_ * b_tgt[i+1, :T-1] + (1-lambda_) * h[i+1, :T-1]
    h = h.at[i, 1:].set(tmp)

In [1422]:
aux = jnp.ones(5)

In [1443]:
jax.lax.select(False, jnp.zeros(4), jnp.ones(4))

Array([1., 1., 1., 1.], dtype=float32)

In [1489]:
jnp.insert(aux, T, 10)

Array([ 1.,  1.,  1.,  1.,  1., 10.], dtype=float32)

In [40]:
# h_tgt = some_tgt
tgt_init = jnp.insert(b_tgt[T-1, :-1], T, 0)
tgt_init

Array([0.5, 0.5, 0.5, 0.5, 0. ], dtype=float32)

In [113]:
b_tgt

Array([[0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5],
       [0.5, 0.5, 0.5, 0.5, 0.5]], dtype=float32)

In [137]:
def get_target(b_tgt, h_tgt, lambda_, T):
    
    h = h_tgt
    stgt_init = jnp.insert(b_tgt[T-1, 1:], T, 0)
    htgt_init = jnp.zeros(T,)
    h_init = (stgt_init, htgt_init)
    
    def f(carry, h):
        next_b_tgt, next_h = carry
        h_tgt, b_tgt = h

        h = lambda_ * next_h + (1-lambda_) * next_b_tgt
        h = jnp.insert(h[:-1], 0, h_tgt[0])
        h = jax.lax.select(h_tgt[0]==1, jnp.zeros(T).at[0].set(1), h)
        
        carry = (b_tgt, h)
        return carry, h

    carry, out = jax.lax.scan(f, h_init, (h_tgt, b_tgt), reverse=True)
    return out

In [138]:
out = get_target(b_tgt, jnp.zeros((T,T)), 0.6, T)
out

Array([[0.        , 0.2       , 0.32000002, 0.39200002, 0.43520004],
       [0.        , 0.2       , 0.32000002, 0.39200002, 0.43520004],
       [0.        , 0.2       , 0.32000002, 0.39200002, 0.39200002],
       [0.        , 0.2       , 0.32000002, 0.32000002, 0.32000002],
       [0.        , 0.2       , 0.2       , 0.2       , 0.2       ]],      dtype=float32)

In [139]:
out = get_target(b_tgt, h_tgt, .6, T)
out

Array([[0.        , 0.2       , 0.32000002, 0.39200002, 0.5648    ],
       [0.        , 0.2       , 0.32000002, 0.60800004, 0.39200002],
       [0.        , 0.2       , 0.68      , 0.32000002, 0.32000002],
       [0.        , 0.8       , 0.2       , 0.2       , 0.2       ],
       [1.        , 0.        , 0.        , 0.        , 0.        ]],      dtype=float32)

In [232]:
# Create some target with death time k=3
h_tgt = pad_to(jnp.eye(3)[::-1], (T, T))

In [233]:
h_tgt

Array([[0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [234]:
out = get_target(b_tgt, h_tgt, 1, T)
out

Array([[0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [243]:
k = 3

In [244]:
h_wgt = jnp.tril(jnp.ones_like(h_tgt), -(h_tgt.shape[0]-1-(k-1)))[::-1]

In [245]:
h_wgt

Array([[1., 1., 1., 0., 0.],
       [1., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [68]:
def survival_curve(xs):
    logits = agent.forward(agent.state.params, xs)
    log_hs = jax.nn.log_sigmoid(logits)
    surv = jnp.exp(jnp.cumsum(log_hs - logits, axis=-1))
    return jnp.insert(surv, 0, 1.0, axis=-1)

In [69]:
s_wgt = survival_curve(xs)[0]

In [148]:
s_wgt

Array([[1.     , 0.5    , 0.25   , 0.125  , 0.0625 , 0.03125],
       [1.     , 0.5    , 0.25   , 0.125  , 0.0625 , 0.03125],
       [1.     , 0.5    , 0.25   , 0.125  , 0.0625 , 0.03125],
       [1.     , 0.5    , 0.25   , 0.125  , 0.0625 , 0.03125],
       [1.     , 0.5    , 0.25   , 0.125  , 0.0625 , 0.03125]],      dtype=float32)

Array([[1., 1., 1., 1., 0.],
       [1., 1., 1., 0., 0.],
       [1., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [235]:
def get_weights(s_wgt, h_wgt, h_tgt, c, lambda_, T):
    
    h = s_wgt
    w_init = s_wgt[T-1]
    w_init = jnp.insert(w_init[:-1], 0, 1.0)
    h_init = (s_wgt[T-1], h_wgt[T-1])
    
    def f(carry, h):
        next_s_wgt, next_h = carry
        h_tgt, h_wgt, s_wgt = h

        h = lambda_ * next_h + (1-lambda_) * next_s_wgt
        value = jax.lax.select(c, 1.0,  h_wgt[0])
        h = jnp.insert(h[:-1], 0, value)
        h = jax.lax.select(h_tgt[0]==1, jnp.zeros(T).at[0].set(1), h)
        
        carry = (s_wgt, h)
        return carry, h

    carry, out = jax.lax.scan(f, h_init, (h_tgt, h_wgt, s_wgt), reverse=True)
    return out

In [250]:
out = get_weights(s_wgt[:, :-1], jnp.ones((T,T)), jnp.zeros((T,T)), True, 0, T)
out

Array([[1.   , 1.   , 0.5  , 0.25 , 0.125],
       [1.   , 1.   , 0.5  , 0.25 , 0.125],
       [1.   , 1.   , 0.5  , 0.25 , 0.125],
       [1.   , 1.   , 0.5  , 0.25 , 0.125],
       [1.   , 1.   , 0.5  , 0.25 , 0.125]], dtype=float32)

In [247]:
h_tgt

Array([[0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [251]:
out = get_weights(s_wgt[:, :-1], h_wgt, h_tgt, False, 0.6, T)
out

Array([[1.        , 1.        , 0.8       , 0.22000001, 0.11000001],
       [1.        , 1.        , 0.2       , 0.1       , 0.05      ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.4       , 0.44000003, 0.22000001, 0.11000001],
       [0.        , 0.4       , 0.2       , 0.1       , 0.05      ]],      dtype=float32)

In [ ]:
# Get hard targets and weights, map get_weigths

In [386]:
h_tgts, h_ws, masks = get_targets_and_masks(xs, ts, cs, True)

In [387]:
s_ws = survival_curve(xs)
s_ws = s_ws[:, :, :-1]

In [388]:
s_ws.shape

(467, 5, 5)

In [389]:
partial_ws = partial(get_weights, lambda_=0.6, T=T)
mapped_ws = jax.vmap(partial_ws, in_axes=(0, 0, 0, 0))  # ()

In [390]:
# s_wgt, h_wgt, h_tgt, c
out = mapped_ws(s_ws, h_ws, h_tgts, cs)

In [391]:
out[29]

Array([[1.        , 1.        , 0.8       , 0.58000004, 0.39800003],
       [1.        , 1.        , 0.8       , 0.58000004, 0.39800003],
       [1.        , 1.        , 0.8       , 0.58000004, 0.39800003],
       [1.        , 1.        , 0.8       , 0.58000004, 0.47000006],
       [1.        , 1.        , 0.8       , 0.70000005, 0.65000004]],      dtype=float32)

In [392]:
ts[29]

Array(1, dtype=int32)

In [393]:
out = get_weights(s_ws[2], h_ws[2], h_tgts[2], cs[2], 0, T)
out

Array([[1.   , 1.   , 0.5  , 0.25 , 0.125],
       [1.   , 1.   , 0.5  , 0.25 , 0.125],
       [1.   , 0.   , 0.   , 0.   , 0.   ],
       [0.   , 1.   , 0.5  , 0.25 , 0.125],
       [0.   , 1.   , 0.5  , 0.25 , 0.125]], dtype=float32)

In [399]:
ts[29]

Array(1, dtype=int32)

In [291]:
# Get soft targets, map get_target

In [292]:
base_tgt = agent.forward(agent.state.params, xs)
s_tgt = jax.nn.sigmoid(base_tgt)

In [293]:
s_tgt.shape

(467, 5, 5)

In [306]:
partial_tgt = partial(get_target, lambda_=.2, T=T)
mapped_tgt = jax.vmap(partial_tgt, in_axes=(0, 0))  # ()

In [307]:
out = mapped_tgt(s_tgt, h_tgts)

In [308]:
out.shape

(467, 5, 5)

In [309]:
out[29]

Array([[0.        , 0.4       , 0.48000002, 0.49600002, 0.49920002],
       [0.        , 0.4       , 0.48000002, 0.49600002, 0.49920002],
       [0.        , 0.4       , 0.48000002, 0.49600002, 0.49600002],
       [0.        , 0.4       , 0.48000002, 0.48000002, 0.48000002],
       [0.        , 0.4       , 0.4       , 0.4       , 0.4       ]],      dtype=float32)

In [325]:
mask[mask_out:]

Array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]], dtype=float32)

In [323]:
jnp.zeros((mask_out, T))

Array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [326]:
t = k
mask = jnp.ones_like(h_tgt)
if t < T:
    mask_out = T - t
    mask = mask.at[mask_out+1:, :].set(jnp.zeros((mask_out, T)))

In [327]:
mask

Array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [328]:
h_tgt

Array([[0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [333]:
ts[22]

Array(1, dtype=int32)

In [334]:
h_tgts[22]

Array([[1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [372]:
def get_single_target_and_mask(seq, t, c, landmark=False):
    h, _ = seq.shape
    target = jnp.zeros((h, h))
    h_ws = jnp.ones((h, h))
    if not c:  # Subject reached terminal state within the horizon.
        target = jnp.eye(t)[::-1]
        target = pad_to(target, shape=(h, h))
        if landmark:
            h_ws = jnp.ones_like(target)
            tt = t.item()
            if tt <= h:
                h_ws = jnp.tril(jnp.ones_like(target), -(h-t.item()))[::-1]
        else:
            t = min(t, seq.shape[0])
            h_ws = jnp.ones((1, t))
            h_ws = pad_to(h_ws, shape=(h, h))
    
    mask = jnp.ones_like(target)
    if t < h:
        mask_out = h - t
        mask = mask.at[t:, :].set(jnp.zeros((mask_out, h)))

    return target, h_ws, mask

In [382]:
get_single_target_and_mask(xs[315], ts[315], cs[315], True)

(Array([[0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0.]], dtype=float32),
 Array([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0.]], dtype=float32),
 Array([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]], dtype=float32))

(Array([315], dtype=int32),)

In [366]:
mask = jnp.ones((T,T))

In [367]:
mask_out = T - 1

In [370]:
mask[1:]

Array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]], dtype=float32)

In [371]:
jnp.zeros((mask_out, T))

Array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [369]:
mask

Array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.]], dtype=float32)